In [ ]:
import pandas as pd

df=pd.read_csv('customer_shopping_behavior.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.isnull().sum()

In [ ]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df.isnull().sum()

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})
df.columns


In [ ]:
labels=['Young Adults', 'Adults','Middle-aged', 'Seniors']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)
df[['age', 'age_group']].head(10)

In [ ]:
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)
df[['frequency_of_purchases', 'purchase_frequency_days']].head(10)

In [ ]:
df[['discount_applied','promo_code_used']].head(10)

In [ ]:
(df['discount_applied'] == df['promo_code_used']).all()

In [ ]:
df = df.drop('promo_code_used', axis=1, errors='ignore')
print("Columns after dropping redundant column:")
print(df.columns.tolist())

In [ ]:
%pip install -r requirements.txt

In [ ]:
import psycopg2
print("psycopg2 version:", psycopg2.__version__)
print("psycopg2 driver loaded successfully")

In [ ]:
import os
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Configure these variables in your environment; never commit database credentials.
username = os.getenv('POSTGRES_USER', 'postgres')
password = os.getenv('POSTGRES_PASSWORD')
database = os.getenv('POSTGRES_DB', 'customer_behavior')
host = os.getenv('POSTGRES_HOST', 'localhost')
port = int(os.getenv('POSTGRES_PORT', '5432'))

if not password:
    raise EnvironmentError(
        'Set POSTGRES_PASSWORD in your environment before loading data into PostgreSQL.'
    )

connection_url = URL.create(
    drivername='postgresql+psycopg2',
    username=username,
    password=password,
    host=host,
    port=port,
    database=database,
)

try:
    engine = create_engine(connection_url)
    with engine.begin() as connection:
        df.to_sql('customer', connection, if_exists='replace', index=False)
    print(f"Data successfully loaded into table 'customer' in database '{database}'.")
except Exception as exc:
    print(f'Could not load data into PostgreSQL: {exc}')
    raise
finally:
    if 'engine' in locals():
        engine.dispose()
